# Task 4: Fine-Tuning BERT on IMDB Dataset

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

## Step 1: Install and Import Required Libraries

In [ ]:
!pip install transformers==4.37.2 accelerate==0.27.2 peft==0.8.2 datasets==2.18.0 huggingface_hub==0.20.3 --no-deps -q

!pip install tokenizers==0.15.2 pyarrow-hotfix sentencepiece scikit-learn matplotlib -q

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Step 2: Load IMDB Dataset

In [ ]:
dataset = load_dataset("imdb")

print(dataset)

## Step 3: Load BERT Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

## Step 4: Tokenize Dataset

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

## Step 5: Create Smaller Training and Testing Samples

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

## Step 6: Load Pretrained BERT Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

## Step 7: Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

## Step 8: Define Evaluation Metrics

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## Step 9: Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
    compute_metrics=compute_metrics
)

## Step 10: Train the Model

In [ ]:
trainer.train()

## Step 11: Evaluate the Model

In [ ]:
predictions = trainer.predict(small_test_dataset)

print(predictions.metrics)

## Step 12: Save the Fine-Tuned Model

In [ ]:
trainer.save_model("bert-imdb-model")

tokenizer.save_pretrained("bert-imdb-model")